# cxr-shift-uncertainty: full run

One run-all notebook. It installs requirements, downloads PneumoniaMNIST and ChestMNIST (224px)
and the pretrained `facebook/convnext-tiny-224` weights, fine-tunes on PneumoniaMNIST, evaluates on
both datasets, runs the strong augmentation ablation, and writes tables and figures to `results/`.

Before running: **Runtime > Change runtime type > GPU** (T4 is enough), then **Runtime > Run all**.

The ChestMNIST 224px file is large (the 112,120 images alone are about 5.6 GB uncompressed),
so the first download takes a while.

In [ ]:
# Settings
REPO_URL = "https://github.com/praneethnarayanan2002-lab/cxr-shift-uncertainty.git"
SMOKE = False          # True: tiny model on synthetic data, finishes in about a minute on CPU
RUN_ABLATION = True    # also train the strong augmentation model
INSTALL = True         # pip install -r requirements.txt
EXTRA_OVERRIDES = []   # e.g. ["train.epochs=5", "data.target.max_samples.test=5000"]

In [ ]:
import os, subprocess, sys
from pathlib import Path

def sh(cmd):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")

# Use the current folder if it already is the repo, otherwise clone it.
if not Path("src/cxrshift").exists():
    if not Path("cxr-shift-uncertainty").exists():
        sh(f"git clone {REPO_URL}")
    os.chdir("cxr-shift-uncertainty")
print("working directory:", os.getcwd())
if INSTALL:
    sh(f"{sys.executable} -m pip install -q -r requirements.txt")

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
elif not SMOKE:
    print("WARNING: no GPU found. Training ConvNeXt-Tiny on CPU will be very slow.")

## 1. Check the label mapping

Prints the label names medmnist reports and how many images map to pneumonia (1) and
negative (0). For ChestMNIST, negatives are images with no finding; images with other findings
and no pneumonia are dropped. This also downloads both datasets.

In [ ]:
BASE_CFG = "configs/smoke.yaml" if SMOKE else "configs/default.yaml"
OUT = "results/smoke_test" if SMOKE else "results"
overrides = " ".join(EXTRA_OVERRIDES)
sh(f"{sys.executable} scripts/check_label_mapping.py --config {BASE_CFG}" + (f" --set {overrides}" if overrides else ""))

## 2. Baseline: fine-tune on PneumoniaMNIST, evaluate on both datasets

Early stopping on source validation AUROC. Temperature, decision threshold and the conformal
quantile are fit on source validation data only. The ChestMNIST test set is never used for fitting.

In [ ]:
sets = " ".join(["run_name=baseline"] + EXTRA_OVERRIDES)
sh(f"{sys.executable} scripts/run_experiment.py --config {BASE_CFG} --set {sets}")

## 3. Ablation: stronger augmentation

Same model and schedule, with extra brightness, contrast, gamma, blur, noise and geometry
augmentation. Histogram matching of target images to the source is evaluated inside every run
as `target_test_histmatch`, so it needs no separate training.

In [ ]:
runs = [f"{OUT}/baseline"]
if RUN_ABLATION:
    sets = " ".join(["run_name=strong_aug", "train.augment=strong"] + EXTRA_OVERRIDES)
    sh(f"{sys.executable} scripts/run_experiment.py --config {BASE_CFG} --set {sets}")
    runs.append(f"{OUT}/strong_aug")
sh(f"{sys.executable} scripts/compare_runs.py {' '.join(runs)} --out {OUT}/comparison.md")

## 4. Results

In [ ]:
from IPython.display import Image, Markdown, display

for run in runs:
    display(Markdown(f"### {run}\n\n" + Path(run, "results_table.md").read_text()))
display(Markdown("### Comparison\n\n" + Path(OUT, "comparison.md").read_text()))

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
conf = pd.read_csv(f"{OUT}/baseline/conformal.csv")
cols = ["dataset", "score", "mondrian", "coverage", "avg_set_size", "coverage_class0", "coverage_class1",
        "frac_singleton", "frac_full", "frac_empty"]
display(conf[cols].round(3))

In [ ]:
for name in ["reliability", "conformal_coverage", "shortcut_check", "gradcam_source_test", "gradcam_target_test"]:
    path = Path(OUT, "baseline", "figures", f"{name}.png")
    if path.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))

## 5. Save the results

Tables and figures are in `results/`. In Colab, the cell below zips them for download (model
weights and per-sample predictions are left out). To put the numbers in the README, copy rows
from `results/baseline/results_table.md` and `results/comparison.md`.

In [ ]:
import shutil
export = Path("results_export")
shutil.rmtree(export, ignore_errors=True)
shutil.copytree(OUT, export, ignore=shutil.ignore_patterns("*.pt", "predictions_*.csv", "smoke_test"))
archive = shutil.make_archive("cxr_shift_results", "zip", export)
print("wrote", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass